### 3.5.4. Gradient Method Convergence


$$
\lim_{k\to\infty}\nabla f(x^k)=0,
\qquad
f(x^{k+1}) \le f(x^k),
\qquad
g^k = \nabla f(x^k)+e^k.
$$


**Explanation:**

Convergence analysis asks what the entire generated sequence tells us, not merely whether one step decreased the cost. Under suitable assumptions, such as boundedness of the iterates, smoothness of the cost, and sufficiently good descent, limit points of gradient methods are stationary. This conclusion is weaker than saying the method finds a global minimum, but it is the correct local target for general nonconvex unconstrained problems. Termination, spacer steps, and gradient errors also matter because real algorithms often use approximate derivatives or stop before exact stationarity.

The limit statement says the gradient norm tends to stationarity along the generated iterates. The decrease inequality compares successive objective values, while $g^k=\nabla f(x^k)+e^k$ reads the computed gradient as the true gradient plus an error term.

For engineering readers, convergence results should be read as contracts between assumptions and conclusions. If the assumptions are violated, the theorem may no longer describe the computation. If they are satisfied, the theorem explains why the method is at least driven toward points satisfying the first-order necessary condition.

**Intuition:**

If an error term in the gradient is smaller than the true gradient in norm, the perturbed direction can remain a descent direction. The proof figure shows how a quadratic upper estimate controls the actual cost decrease when the gradient is Lipschitz, a concept naturally expressed using [matrix norm](../../01_Foundations/01_Linear_Algebra/03_Matrix/16_matrix_norm_inequality.ipynb)-type bounds. The code cell then shows gradient [norms](../../01_Foundations/01_Linear_Algebra/02_Vector/06_vector_norm.ipynb) decreasing along a stable gradient iteration.

<img src="../../Figures/030504_berts_fig_1_2_9_gradient_error_direction.jpeg" alt="Bertsekas Figure 1.2.9: descent direction under gradient error" style="width: 60%; height: auto;">

<img src="../../Figures/030504_berts_fig_1_2_10_convergence_majorization.jpeg" alt="Bertsekas Figure 1.2.10: quadratic majorization used in a convergence proof" style="width: 60%; height: auto;">


**Numerical Example:**

The convergence-majorization figure supports the idea that repeated descent drives the gradient norm down. Let
$$
Q=\begin{bmatrix}3&0\\0&1\end{bmatrix},\qquad x^0=\begin{bmatrix}3\\-2\end{bmatrix},\qquad \alpha=0.25.
$$

At the first iterate,
$$
g^0=Qx^0=\begin{bmatrix}9\\-2\end{bmatrix},
\qquad \|g^0\|=\sqrt{9^2+(-2)^2}=\sqrt{85}\approx 9.22.
$$

The update gives
$$
x^1=x^0-0.25g^0
=\begin{bmatrix}3\\-2\end{bmatrix}-0.25\begin{bmatrix}9\\-2\end{bmatrix}
=\begin{bmatrix}0.75\\-1.5\end{bmatrix}.
$$

Then
$$
g^1=Qx^1=\begin{bmatrix}2.25\\-1.5\end{bmatrix},
\qquad \|g^1\|=\sqrt{2.25^2+(-1.5)^2}\approx 2.70.
$$

One more step gives $x^2=(0.1875,-1.125)^{\mathsf T}$ and
$$
\|g^2\|=\sqrt{0.5625^2+(-1.125)^2}\approx 1.26,
$$
so the first-order residual is decreasing.


In [1]:
import sympy as sp

x_1, x_2 = sp.symbols("x_1 x_2")
decision_variable = sp.Matrix([x_1, x_2])
Q = sp.Matrix([[3, 0], [0, 1]])
objective = (decision_variable.T * Q * decision_variable)[0] / 2
gradient = sp.Matrix([sp.diff(objective, variable) for variable in decision_variable])

initial_point = sp.Matrix([3, -2])
stepsize = sp.Rational(1, 4)

points = [initial_point]
for iteration in range(2):
    gradient_at_point = gradient.subs({x_1: points[-1][0], x_2: points[-1][1]})
    points.append(points[-1] - stepsize * gradient_at_point)

gradient_norms = [gradient.subs({x_1: point[0], x_2: point[1]}).norm() for point in points]

print("gradient_norms ="); sp.pprint(sp.Matrix(gradient_norms).T)
print("points =", [point.T.tolist()[0] for point in points])

gradient_norms =
⎡     3⋅√13  9⋅√5⎤
⎢√85  ─────  ────⎥
⎣       4     16 ⎦
points = [[3, -2], [3/4, -3/2], [3/16, -9/8]]


**Equivalent `casadi` implementation:**

In [2]:
import casadi as ca

decision_variable = ca.SX.sym("x", 2)
Q = ca.DM([[3, 0], [0, 1]])
objective = 0.5 * ca.bilin(Q, decision_variable, decision_variable)
gradient_function = ca.Function("grad", [decision_variable], [ca.gradient(objective, decision_variable)])

initial_point = ca.DM([3, -2])
stepsize = 0.25

points = [initial_point]
for iteration in range(2):
    gradient = gradient_function(points[-1])
    points.append(points[-1] - stepsize * gradient)

gradient_norms = [float(ca.norm_2(gradient_function(point))) for point in points]

print("gradient_norms =", gradient_norms)
print("points =", [point.full().flatten().tolist() for point in points])

gradient_norms = [9.219544457292887, 2.704163456597992, 1.2577882373436318]
points = [[3.0, -2.0], [0.75, -1.5], [0.1875, -1.125]]


**References:**

[📘 Bertsekas, D. P. (1999). *Nonlinear Programming* (2nd ed.), Chapter 1 "Unconstrained Optimization". Athena Scientific.](https://www.athenasc.com/nonlinbook.html)

---

[⬅️ Previous: Stepsize Rules](./03_stepsize_rules.ipynb) | [Next: Strong Convexity and Sublevel Geometry ➡️](./05_strong_convexity_and_sublevel_geometry.ipynb)

---
